<div style="font-size: 1em; display: flex; align-items: center; gap: 8px; padding: 8px 16px; background: #F8F9FA; border-bottom: 2px solid #E0E0E0; margin: 0; line-height: 1">
    <img src="https://cdn.simpleicons.org/databricks/FF3621" width="24" height="24"/>
    <div style="color: #666">
        <span style="font-weight: bold; color: #333">Databricks Performance Optimization
</span>
        <span style="margin-left: 8px; color: #999">|</span>
        <span style="margin-left: 8px">08 Demo - Python, Driver and UDF Pitfalls</span>
    </div>
</div>

![DB Academy](https://files.training.databricks.com/binder/prod_main/databricks-performance-optimization-en_us-3.0.1/images/20261106T135523Z/Databricks Performance Optimization/Includes/Images/common/databricks_academy_logo.png)

# Demo - Python, Driver and UDF Pitfalls
#### Duration: 20 minutes
## Overview

Part 7 covered four distributed slow-downs: shuffle, skew, spill, and explosion. This part covers the fifth pattern: work that stays on the driver instead of reaching the executors.

When code runs on the driver instead of across the executors, or when a Python UDF forces the query off Photon, the cluster does little useful work while the wall clock accumulates.

You will build a mental model of the driver-versus-executor split, name the driver traps that serialize your job, and rank the UDF options from fastest to slowest. 

Then you run the same transform two ways and read the difference in the profile.

> Serverless-first: this demo runs on a serverless notebook. Serverless and Photon optimize distributed work, but cannot parallelize code that funnels everything to the driver.

## Learning Objectives

By the end of this demo, you will be able to:

1. **Explain** the driver-versus-executor split and why some code silently runs single-threaded on the driver.
2. **Identify** the driver traps (`collect()`, `toPandas()`, `display()` in production, single-threaded pandas on a multi-node cluster, plan bloat) and their symptoms.
3. **Rank** the UDF options from fastest to slowest, and explain what each step down costs in serialization and Photon coverage.
4. **Verify** that a Python UDF leaves Photon by reading the Query Profile and `total_task_duration_ms`, and connect that to UDF choice.

<div style="border-left:4px solid #FFAB00;background:rgba(255,171,0,0.08);padding:14px 18px;border-radius:4px;margin:16px 0;font-family:sans-serif;">
  <strong style="display:block;color:#B8860B;margin-bottom:6px;font-size:15pt;">Pre-Run Timing Note</strong>
  <div style="color:#0b2026;font-size:14pt;line-height:1.55;">
    This notebook reads from <code>system_vw.query.history</code>, which lags live activity by <strong>5 to 10 minutes</strong>. Run the query cells first, then continue with the explanatory sections. By the time you reach the system-table cells, the rows will have landed. If a cell returns no rows, wait a few minutes and re-run it.
  </div>
</div>

## REQUIRED - SELECT A COMPUTE ENVIRONMENT

<div style="
  border-left: 4px solid #f44336;
  background: #ffebee;
  padding: 14px 18px;
  border-radius: 4px;
  margin: 16px 0;
">
  <strong style="display:block; color:#c62828; margin-bottom:6px; font-size: 1.1em;">Select Serverless Compute</strong>
  <div style="color:#333;">

Before starting this notebook, select the required compute environment listed below.

- **Serverless Compute, Version 5**

![Serverless Select](https://files.training.databricks.com/binder/prod_main/databricks-performance-optimization-en_us-3.0.1/images/20261106T135523Z/Databricks Performance Optimization/Includes/Images/common/select-serverless.png)
<br></br>
  - How to select an environment version:
[AWS](https://docs.databricks.com/aws/en/compute/serverless/dependencies#-select-an-environment-version) |
[Azure](https://learn.microsoft.com/en-us/azure/databricks/compute/serverless/dependencies#-select-an-environment-version) |
[GCP](https://docs.databricks.com/gcp/en/compute/serverless/dependencies#-select-an-environment-version)
  </div>
</div>

## A. Classroom Setup

Run the setup below. It reuses the shared retail demo tables and stages helpers:

- `price_band_py`, `base_line_items()`, `builtin_tier_col()`, `time_count()`.
- `variant_cost_history()` for reading `total_task_duration_ms` from `system_vw.query.history`.

In [ ]:
%run ./Includes/Classroom-Setup-08

<div class="cw">
<style>
.cw { font-family: sans-serif; max-width: 1100px; margin: 0 auto; color: #0b2026; display: flex; flex-direction: column; gap: 20px; }
.cw * { box-sizing: border-box; }
.ds-banner { display: flex; align-items: center; gap: 20px; width: 100%; border-radius: 10px; padding: 24px 28px; background: var(--bnr-bg); border-left: 8px solid var(--bnr-accent); box-shadow: 0 2px 10px rgba(27,49,57,0.08); }
.ds-banner-icon { width: 60px; height: 60px; border-radius: 50%; flex-shrink: 0; background: var(--bnr-accent); color: var(--bnr-icon-fg); font-size: 30pt; font-weight: 700; line-height: 1; display: flex; align-items: center; justify-content: center; }
.ds-banner-title { font-size: 22pt; font-weight: 700; color: var(--bnr-title); line-height: 1.2; margin-bottom: 4px; }
.ds-banner-tagline { font-size: 15pt; color: var(--bnr-tagline); line-height: 1.4; }
.ds-banner.info { --bnr-bg: #1B5162; --bnr-accent: #4299E0; --bnr-icon-fg: #FFFFFF; --bnr-title: #FFFFFF; --bnr-tagline: #F9F7F4; }
.ds-row { display: flex; gap: 24px; align-items: stretch; flex-wrap: wrap; }
.ds-row > * { flex: 1; min-width: 240px; }
.ds-card { background: #F9F7F4; border-radius: 8px; box-shadow: 0 2px 8px rgba(27,49,57,0.06); padding: 20px; border-top: 8px solid var(--accent, #4299E0); }
.blue{--accent:#4299E0;} .green{--accent:#00A972;} .coral{--accent:#FF5F46;} .amber{--accent:#FFAB00;} .red{--accent:#98102A;}
.ds-card-title { font-size: 18pt; font-weight: 700; line-height: 1.25; margin: 0 0 8px; }
.ds-card ul { margin: 8px 0 0; padding-left: 20px; font-size: 14pt; line-height: 1.55; color: #0b2026; }
.ds-card li { margin-bottom: 10px; }
.ds-card li:last-child { margin-bottom: 0; }
</style>

<div class="ds-banner info">
  <div class="ds-banner-icon">5</div>
  <div>
    <div class="ds-banner-title">The 5th pattern: the work never got distributed</div>
    <div class="ds-banner-tagline">Shuffle, skew, spill and explosion are about data moving. 
    <br />
    This is about work that never left the driver, or never entered Photon.</div>
  </div>
</div>

<div class="ds-row">
  <div class="ds-card blue">
    <div class="ds-card-title">The first 4 patterns</div>
    <ul>
      <li>Shuffle, skew, spill, explosion.</li>
      <li>All are symptoms of <strong>distributed</strong> work under stress.</li>
      <li>Loud in the profile: Exchange operators, straggler tasks, big row counts.</li>
    </ul>
  </div>
  <div class="ds-card coral">
    <div class="ds-card-title">The 5th</div>
    <ul>
      <li>The work runs on the <strong>driver</strong> or a Python UDF blocks <strong>Photon</strong>.</li>
      <li>Cores sit idle while one thread grinds.</li>
      <li>Quiet in the profile: the clock runs long with little to show.</li>
    </ul>
  </div>
  <div class="ds-card green">
    <div class="ds-card-title">The fix:</div>
    <ul>
      <li>Keep the work <strong>distributed</strong> and keep it <strong>in Photon</strong>.</li>
      <li>Prefer DataFrame and SQL expressions over row-by-row Python.</li>
      <li>Serverless autoscaling only helps work that is already distributed.</li>
    </ul>
  </div>
</div>
</div>

##### EXPAND FOR ADDITIONAL DETAILS

<details>

### Why this pattern is different
- The four patterns in Part 7 are all failure modes of distributed execution. You read them from Exchange operators, straggler tasks, spill bytes, and row-count jumps. They are loud.
- This fifth pattern is the opposite. The symptom is that the cluster is barely working. One core on the driver is busy, the executors are idle, and the wall clock runs long anyway. Databricks' own "top Spark mistakes" list puts collecting to the driver at number one for a reason.
- Two root causes sit under it: (1) code that pulls data to the driver and processes it there (`collect()`, `toPandas()`, single-threaded pandas or scikit-learn), and (2) code that stays distributed but forces Photon to fall back to the slower JVM path, which is what a Python UDF does.

### The through-line for this part
- The fix in both cases: keep work on the executors (do not funnel it to the driver) and keep it in Photon (prefer built-ins and SQL to Python UDFs).
- Serverless autoscaling and Photon optimize distributed work. They cannot parallelize a `collect()` or vectorize a Python loop, so the code must express the work as DataFrame or SQL operations for the platform to help.

</details>

## B. Driver Traps
<br/>
<div class="cw">
<style>
.cw { font-family: sans-serif; max-width: 1100px; margin: 0 auto; color: #0b2026; display: flex; flex-direction: column; gap: 20px; }
.cw * { box-sizing: border-box; }
.ds-banner { display: flex; align-items: center; gap: 20px; width: 100%; border-radius: 10px; padding: 24px 28px; background: var(--bnr-bg); border-left: 8px solid var(--bnr-accent); box-shadow: 0 2px 10px rgba(27,49,57,0.08); }
.ds-banner-icon { width: 60px; height: 60px; border-radius: 50%; flex-shrink: 0; background: var(--bnr-accent); color: var(--bnr-icon-fg); font-size: 30pt; font-weight: 700; line-height: 1; display: flex; align-items: center; justify-content: center; }
.ds-banner-title { font-size: 22pt; font-weight: 700; color: var(--bnr-title); line-height: 1.2; margin-bottom: 4px; }
.ds-banner-tagline { font-size: 15pt; color: var(--bnr-tagline); line-height: 1.4; }
.ds-banner.info { --bnr-bg: #1B5162; --bnr-accent: #4299E0; --bnr-icon-fg: #FFFFFF; --bnr-title: #FFFFFF; --bnr-tagline: #F9F7F4; }
</style>

<div class="ds-banner info">
  <div class="ds-banner-icon">i</div>
  <div>
    <div class="ds-banner-title">One driver, many executors</div>
    <div class="ds-banner-tagline">The driver plans and coordinates and the executors do the parallel work. The problems start when data processing lands on the driver.</div>
  </div>
</div>

<div style="background:#FFFFFF;border-radius:10px;padding:18px;box-shadow:0 2px 8px rgba(27,49,57,0.06);">
<svg width="100%" viewBox="0 0 860 300" role="img" style="font-family: sans-serif;">
  <defs>
    <marker id="dr-arrow" viewBox="0 0 10 10" refX="8" refY="5" markerWidth="6" markerHeight="6" orient="auto-start-reverse"><path d="M2 1L8 5L2 9" fill="none" stroke="#1B3139" stroke-width="1.5" stroke-linecap="round" stroke-linejoin="round"/></marker>
    <marker id="dr-arrow-red" viewBox="0 0 10 10" refX="8" refY="5" markerWidth="6" markerHeight="6" orient="auto-start-reverse"><path d="M2 1L8 5L2 9" fill="none" stroke="#98102A" stroke-width="1.5" stroke-linecap="round" stroke-linejoin="round"/></marker>
  </defs>
  <!-- driver -->
  <rect x="40" y="110" width="180" height="90" rx="10" fill="#1B5162" />
  <text x="130" y="146" text-anchor="middle" font-size="19" font-weight="700" fill="#FFFFFF">Driver</text>
  <text x="130" y="170" text-anchor="middle" font-size="14" fill="#F9F7F4">plans and coordinates</text>
  <text x="130" y="188" text-anchor="middle" font-size="14" fill="#F9F7F4">one process</text>
  <!-- healthy distribute arrows -->
  <path d="M220 135 L300 70" fill="none" stroke="#00A972" stroke-width="2" marker-end="url(#dr-arrow)"/>
  <path d="M220 155 L300 155" fill="none" stroke="#00A972" stroke-width="2" marker-end="url(#dr-arrow)"/>
  <path d="M220 175 L300 240" fill="none" stroke="#00A972" stroke-width="2" marker-end="url(#dr-arrow)"/>
  <text x="250" y="120" text-anchor="middle" font-size="13" fill="#00A972" font-weight="700">tasks</text>
  <!-- executors -->
  <rect x="305" y="40" width="200" height="58" rx="8" fill="#00A972" fill-opacity="0.12" stroke="#00A972" stroke-width="1.5"/>
  <text x="405" y="66" text-anchor="middle" font-size="16" font-weight="700" fill="#0b2026">Executor 1</text>
  <text x="405" y="85" text-anchor="middle" font-size="14" fill="#618794">many cores, parallel</text>
  <rect x="305" y="126" width="200" height="58" rx="8" fill="#00A972" fill-opacity="0.12" stroke="#00A972" stroke-width="1.5"/>
  <text x="405" y="152" text-anchor="middle" font-size="16" font-weight="700" fill="#0b2026">Executor 2</text>
  <text x="405" y="171" text-anchor="middle" font-size="14" fill="#618794">many cores, parallel</text>
  <rect x="305" y="212" width="200" height="58" rx="8" fill="#00A972" fill-opacity="0.12" stroke="#00A972" stroke-width="1.5"/>
  <text x="405" y="238" text-anchor="middle" font-size="16" font-weight="700" fill="#0b2026">Executor 3</text>
  <text x="405" y="257" text-anchor="middle" font-size="14" fill="#618794">many cores, parallel</text>
  <!-- collect() pulls everything back -->
  <path d="M505 69 C 590 90, 560 150, 232 150" fill="none" stroke="#98102A" stroke-width="2" stroke-dasharray="6 4" marker-end="url(#dr-arrow-red)"/>
  <path d="M505 241 C 600 210, 560 165, 232 160" fill="none" stroke="#98102A" stroke-width="2" stroke-dasharray="6 4" marker-end="url(#dr-arrow-red)"/>
  <rect x="560" y="120" width="270" height="72" rx="8" fill="#98102A" fill-opacity="0.08" stroke="#98102A" stroke-width="1.5"/>
  <text x="695" y="150" text-anchor="middle" font-size="16" font-weight="700" fill="#98102A">collect() / toPandas()</text>
  <text x="695" y="172" text-anchor="middle" font-size="14" fill="#0b2026">funnels all rows back to the driver</text>
</svg>
</div>

<div style="border-left:4px solid #98102A;background:rgba(152,16,42,0.08);padding:14px 18px;border-radius:4px;">
  <strong style="display:block;color:#98102A;margin-bottom:6px;font-size:15pt;">The tell</strong>
  <div style="color:#0b2026;font-size:14pt;line-height:1.55;">Executors idle, one driver thread busy, and the wall clock long. 
  <br /><br />
  The Query Profile is quiet because the heavy work happened in Python on the driver, off the engine entirely.</div>
</div>
</div>

##### EXPAND FOR ADDITIONAL DETAILS

<details>

### The driver versus executor split
- The **driver** is a single process. It builds the query plan, schedules tasks, and collects final results. It is not where bulk data processing is supposed to happen.
- The **executors** are where the parallel work runs, many cores across one or more nodes. Distributed transformations (`select`, `filter`, joins, aggregations) run here, in parallel, in Photon.
- The healthy flow is left to right: the driver hands tasks to the executors and they do the work. The trap is the dashed red path: code that pulls all the rows back to the driver and processes them there, single-threaded, in plain Python.

### Why serverless does not save you here
- Serverless autoscaling adds executors when there is distributed work to spread. A `collect()` gives it nothing to spread: the work is one Python thread on the driver. More executors sit idle.
- This is the pattern that looks "quiet" in the profile. There is no big shuffle, no spill, no straggler, just a long stretch of time where the engine was waiting on the driver.

</details>

### B1. The Driver Traps
<br/>
<div class="rc-wrap">
<style>
.rc-wrap { width: 100%; display: flex; flex-direction: column; align-items: center; gap: 18px; font-family: sans-serif; }
.rc-wrap * { box-sizing: border-box; }
.rc-hint { width: 100%; max-width: 1100px; text-align: center; font-size: 14pt; color: #5E7077; font-style: italic; }
.rc-hint strong { color: #FF5F46; font-style: normal; font-weight: 700; }
.rc-row { display: block; white-space: nowrap; font-size: 0; width: 100%; max-width: 1100px; }
.rc-box { display: inline-block; width: 32%; margin-right: 2%; min-height: 96px; background: #F9F7F4; border-top: 8px solid transparent; border-left: 2px solid transparent; border-right: 2px solid transparent; border-bottom: 2px solid transparent; border-radius: 8px; padding: 16px 14px; text-align: center; cursor: pointer; user-select: none; vertical-align: top; transition: transform 0.12s; box-shadow: 0 2px 8px rgba(27,49,57,0.06); white-space: normal; }
.rc-box:last-child { margin-right: 0; }
.rc-box:hover { transform: translateY(-2px); }
.rc-box.active { background: #fff; border-left-color: var(--pc); border-right-color: var(--pc); border-bottom-color: var(--pc); }
.rc-label { display: block; font-weight: 700; font-size: 14pt; letter-spacing: 1px; text-transform: uppercase; line-height: 1.2; pointer-events: none; }
.rc-title { display: block; font-size: 14pt; font-weight: 700; color: #0b2026; line-height: 1.3; margin-top: 6px; pointer-events: none; font-family: Menlo, Consolas, monospace; }
.rc-detail-wrap { overflow: hidden; max-height: 0; opacity: 0; transition: max-height 0.35s ease, opacity 0.28s ease, margin-top 0.28s ease; margin-top: 0; width: 100%; max-width: 1100px; }
.rc-detail-wrap.open { max-height: 700px; opacity: 1; margin-top: 12px; }
.rc-detail-card { background: #F9F7F4; border-radius: 10px; padding: 22px 26px; border-top: 7px solid #ccc; box-shadow: 0 2px 8px rgba(27,49,57,0.06); }
.rc-detail-title { font-size: 17pt; font-weight: 700; color: #0b2026; margin-bottom: 12px; line-height: 1.3; }
.rc-detail-body { font-size: 14pt; color: #0b2026; line-height: 1.55; }
.rc-detail-body ul { margin: 0; padding-left: 22px; }
.rc-detail-body li { margin-bottom: 8px; }
.rc-detail-body li:last-child { margin-bottom: 0; }
.rc-detail-body code { font-family: Menlo, Consolas, monospace; background: #EEEDE9; padding: 1px 5px; border-radius: 3px; }
</style>

<div class="rc-hint">Three ways real work lands on the driver. <strong>Click each trap</strong> for the symptom.</div>

<div class="rc-row">
  <span class="rc-box" data-id="0" onclick="rcSelect(0)" style="--pc:#98102A; border-top-color:#98102A;">
    <span class="rc-label" style="color:#98102A;">Pull to driver</span>
    <span class="rc-title">collect() / toPandas()</span>
  </span><span class="rc-box" data-id="1" onclick="rcSelect(1)" style="--pc:#FFAB00; border-top-color:#FFAB00;">
    <span class="rc-label" style="color:#B8860B;">Single-threaded</span>
    <span class="rc-title">pandas / scikit-learn</span>
  </span><span class="rc-box" data-id="2" onclick="rcSelect(2)" style="--pc:#4299E0; border-top-color:#4299E0;">
    <span class="rc-label" style="color:#4299E0;">Interactive habits</span>
    <span class="rc-title">display() / repeated count()</span>
  </span>
</div>

<div class="rc-detail-wrap" id="rc-detail-wrap">
  <div class="rc-detail-card" id="rc-detail-card"></div>
</div>
</div>

<script>
var REVEAL = [
  { title: "collect() and toPandas()", color: "#98102A", text: "<ul><li>Both move every row to the driver's memory.</li><li>Fine for a handful of summary rows, dangerous for a fact table.</li><li>Symptom: a long pause, one busy thread, idle executors. At scale the driver can run out of memory entirely.</li></ul>" },
  { title: "pandas or scikit-learn on the driver", color: "#FFAB00", text: "<ul><li>After a <code>collect()</code>, the pandas or sklearn work runs on one driver core.</li><li>The multi-node cluster you are paying for sits idle.</li><li>Symptom: one busy core, flat executor metrics.</li></ul>" },
  { title: "display() and repeated count()", color: "#4299E0", text: "<ul><li>Great while exploring, costly left in a production path.</li><li>Each <code>count()</code> re-runs the plan up to that point.</li><li>Symptom: the same scan and shuffle repeated several times.</li></ul>" }
];
var rcCurrent = null;
function rcSelect(id) {
  var wrap = document.getElementById('rc-detail-wrap');
  var card = document.getElementById('rc-detail-card');
  var d = REVEAL[id];
  document.querySelectorAll('.rc-box').forEach(function(b) {
    b.classList.toggle('active', parseInt(b.dataset.id) === id && rcCurrent !== id);
  });
  if (rcCurrent === id) { wrap.classList.remove('open'); rcCurrent = null; return; }
  rcCurrent = id;
  card.style.borderTopColor = d.color;
  card.innerHTML = '<div class="rc-detail-title" style="color:' + d.color + ';">' + d.title + '</div><div class="rc-detail-body">' + d.text + '</div>';
  wrap.classList.add('open');
}
</script>

##### EXPAND FOR ADDITIONAL DETAILS

<details>

### collect() and toPandas()
- Both pull the entire result set into the driver's memory. The rule of thumb: only collect what a human or a small config could reasonably read. A few hundred summary rows is fine. A fact table is not.
- At scale, the driver can exhaust memory (on classic compute this kills the process outright). Even when memory holds, the cluster sits idle while one thread processes the data. The old reflex was to up-size the driver so a big `collect()` survives. The fix is to keep the aggregation distributed and collect only the small final result.

### Single-threaded pandas or scikit-learn
- A common shape is `collect()` or `toPandas()` followed by pandas or scikit-learn code. That processing runs on one core of the driver. On a multi-node cluster, every executor sits idle while one thread grinds.
- When you genuinely need pandas semantics at scale, use the pandas API on Spark (`pyspark.pandas`) or a pandas UDF, both of which distribute. When you need Spark-native ML, use `pyspark.ml`. The point is to keep the compute on the executors.

### display() and repeated count()
- `display()` and `count()` are exploration tools. Left in a scheduled job, they add real work. Each `count()` is an action that re-executes the plan up to that point, so a handful of "just checking" counts can re-scan and re-shuffle the same data several times.
- In production, compute what you need once, write it, and let downstream steps read the result. Save the interactive calls for the notebook you are developing in.

</details>

### B2. Lazy Evaluation

Spark is lazy: transformations build a plan and an action runs it. 

If you build the plan carelessly it will bloat, which slows planning and can defeat Photon. 

Here are two culprits:
<div class="cw" style="margin-top:20px">
<style>
.cw { font-family: sans-serif; max-width: 1100px; margin: 0 auto; color: #0b2026; }
.cw * { box-sizing: border-box; }
.ds-row { display: flex; gap: 24px; align-items: stretch; flex-wrap: wrap; }
.ds-row > * { flex: 1; min-width: 240px; }
.ds-card { background: #F9F7F4; border-radius: 8px; box-shadow: 0 2px 8px rgba(27,49,57,0.06); padding: 20px; border-top: 8px solid var(--accent, #4299E0); }
.green{--accent:#00A972;} .coral{--accent:#FF5F46;}
.dd-head { font-size: 20pt; font-weight: 700; margin: 4px 0 12px; color: var(--accent); }
.ds-card code { background: #EEEDE9; padding: 2px 6px; border-radius: 3px; font-size: 14pt; color: #0b2026; font-family: Menlo, Consolas, monospace; }
.ds-card p { font-size: 14pt; line-height: 1.55; margin: 0 0 10px; }
.ds-card ul { margin: 8px 0 0; padding-left: 20px; font-size: 14pt; line-height: 1.55; color: #0b2026; }
.ds-card li { margin-bottom: 10px; }
</style>

<div class="ds-row">
  <div class="ds-card coral">
    <div class="dd-head">&times; DON'T</div>
    <p>Add columns one at a time in a Python loop:</p>
    <p><code>for c in cols:<br/>&nbsp;&nbsp;df = df.withColumn(c, expr(c))</code></p>
    <ul>
      <li>Each call adds a projection layer, so the plan grows with the loop.</li>
      <li><code>dropDuplicates()</code> across every column shuffles the full row width.</li>
    </ul>
  </div>
  <div class="ds-card green">
    <div class="dd-head">&check; DO</div>
    <p>Add them in one projection:</p>
    <p><code>df = df.withColumns({...})</code> or one <code>select(...)</code></p>
    <ul>
      <li>One projection layer, a flatter plan, faster to compile.</li>
      <li>Dedupe on the <strong>key columns</strong> that define a duplicate, not all columns.</li>
    </ul>
  </div>
</div>
</div>

##### EXPAND FOR ADDITIONAL DETAILS

<details>

### withColumn in a loop
- Every `withColumn` call adds another projection node to the logical plan. In a loop over many columns, the plan grows long and the optimizer spends real time compiling it before any data moves. On a wide transform this can dominate a short query.
- Build all the new columns in a single `withColumns({...})` (one map of name to expression) or a single `select`. One projection layer, a flatter plan, and less to compile. The result is identical and the plan is cheaper.

### dropDuplicates across all columns
- `dropDuplicates()` with no argument keys on every column, so it shuffles the full row width to compare entire rows. That is a wide, expensive shuffle.
- Deduplicate on the columns that actually define a duplicate, for example `dropDuplicates(["order_id"])`. You shuffle far less and you say what you mean.
- You can confirm a bloated plan with `df.explain()`: a long stack of projections or an unexpected wide shuffle is the signal. This is the same "measure, do not guess" habit from Part 2, applied to the plan instead of the profile.

</details>

## C. UDF Options
<br/>
<div id="udfFlipRoot"></div>

<!-- Hidden back-face content (pre-escaped, read by JS) -->
<div id="udf-back-1" style="display:none;">
<p style="margin:0 0 10px;font-size:13pt;line-height:1.5;">Spark SQL functions and higher-order functions (<code>transform</code>, <code>aggregate</code> for arrays) execute inside Photon with no Python involved. Before writing any UDF, check whether a built-in already does it. The library includes string, date, math, array, map, and JSON functions.</p>
<pre style="background:#1B5162;color:#F9F7F4;border-radius:6px;padding:12px;font-size:11.5pt;line-height:1.45;overflow-x:auto;margin:0;white-space:pre-wrap;word-break:break-word;"><code># No UDF needed. Runs fully in Photon.
from pyspark.sql import functions as F

df.withColumn("tier", F.concat(
    F.upper(F.substring("category", 1, 3)),
    F.lit("-"),
    F.when(F.col("list_price") &gt; 500, "A")
     .when(F.col("list_price") &gt; 100, "B")
     .otherwise("C")
))</code></pre>
</div>

<div id="udf-back-2" style="display:none;">
<p style="margin:0 0 10px;font-size:13pt;line-height:1.5;">The same Python function as a pickled UDF, but Arrow batches the data crossing between the JVM and the Python worker instead of pickling one row at a time. The Python still runs off Photon, but serialization cost drops sharply.</p>
<pre style="background:#1B5162;color:#F9F7F4;border-radius:6px;padding:12px;font-size:11.5pt;line-height:1.45;overflow-x:auto;margin:0;white-space:pre-wrap;word-break:break-word;"><code>from pyspark.sql.functions import udf
from pyspark.sql.types import StringType

def price_band(category, list_price):
    prefix = category[:3].upper()
    band = "A" if list_price &gt; 500 else "B" if list_price &gt; 100 else "C"
    return f"{prefix}-{band}"

# useArrow=True is the only difference from a pickled UDF.
tier_udf = udf(price_band, StringType(), useArrow=True)

df.withColumn("tier", tier_udf("category", "list_price"))</code></pre>
<p style="margin:10px 0 0;font-size:12pt;color:#618794;">On serverless env v5, Arrow is locked on, so a plain <code>udf(...)</code> already uses Arrow batching (this rung). On classic DBR 17.3, Arrow is opt-in: you must pass <code>useArrow=True</code> or the UDF falls to rung 4.</p>
</div>

<div id="udf-back-3" style="display:none;">
<p style="margin:0 0 10px;font-size:13pt;line-height:1.5;">Operates on a <code>pandas.Series</code> per batch, not row by row. Also Arrow-based under the hood. The function signature takes and returns Series objects, so you can use vectorized NumPy or pandas operations inside.</p>
<pre style="background:#1B5162;color:#F9F7F4;border-radius:6px;padding:12px;font-size:11.5pt;line-height:1.45;overflow-x:auto;margin:0;white-space:pre-wrap;word-break:break-word;"><code>import pandas as pd
from pyspark.sql.functions import pandas_udf

@pandas_udf("string")
def price_band_pandas(category: pd.Series, list_price: pd.Series) -&gt; pd.Series:
    prefix = category.str[:3].str.upper()
    band = pd.cut(list_price, bins=[0, 100, 500, float("inf")], labels=["C", "B", "A"])
    return prefix + "-" + band.astype(str)

df.withColumn("tier", price_band_pandas("category", "list_price"))</code></pre>
<p style="margin:10px 0 0;font-size:12pt;color:#618794;">Reach for this when the work is genuinely vectorized or needs a pandas/NumPy library call where the batch model pays off.</p>
</div>

<div id="udf-back-4" style="display:none;">
<p style="margin:0 0 10px;font-size:13pt;line-height:1.5;">Serializes and deserializes every row between the JVM and Python one at a time. Forces Photon to fall back to the JVM for that operator. This is what a plain <code>udf(...)</code> gives you on DBR 17.3 because Arrow is opt-in.</p>
<pre style="background:#1B5162;color:#F9F7F4;border-radius:6px;padding:12px;font-size:11.5pt;line-height:1.45;overflow-x:auto;margin:0;white-space:pre-wrap;word-break:break-word;"><code>from pyspark.sql.functions import udf
from pyspark.sql.types import StringType

def price_band(category, list_price):
    prefix = category[:3].upper()
    band = "A" if list_price &gt; 500 else "B" if list_price &gt; 100 else "C"
    return f"{prefix}-{band}"

# No useArrow keyword. On DBR 17.3 this is pickled row-at-a-time.
tier_udf = udf(price_band, StringType())

df.withColumn("tier", tier_udf("category", "list_price"))</code></pre>
<p style="margin:10px 0 0;font-size:12pt;color:#618794;">The Python function is identical to the Arrow version. The only difference is the missing <code>useArrow=True</code>, which drops you from option 2 to option 4 on classic DBR 17.3. On serverless env v5 this distinction does not apply because Arrow is always on.</p>
</div>

<script>
(function() {
    var cards = [
        { num: "1", title: "Built-in functions", subtitle: "Spark SQL and higher-order functions. Run fully in Photon, no Python round trip.", badge: "FASTEST", color: "#00A972", backId: "udf-back-1" },
        { num: "2", title: "Arrow-optimized Python UDF", subtitle: "Same Python function, but Arrow batches rows between JVM and Python instead of pickling them one at a time. useArrow=True. Leaves Photon.", badge: "ARROW", color: "#4299E0", backId: "udf-back-2" },
        { num: "3", title: "pandas UDF", subtitle: "Operates on a pandas Series per batch, not row by row. Also Arrow-based. Use when you need vectorized pandas or NumPy operations inside the function.", badge: "VECTORIZED", color: "#FFAB00", backId: "udf-back-3" },
        { num: "4", title: "Pickled Python UDF", subtitle: "Row-at-a-time pickle serialization. Leaves Photon for the JVM. Default for plain udf(...) on classic DBR 17.3.", badge: "SLOWEST", color: "#98102A", backId: "udf-back-4" }
    ];

    var flippedCards = {};

    function render() {
        var root = document.getElementById('udfFlipRoot');
        var html = '';
        html += '<div style="font-family: sans-serif; width: 100%; margin: 0; padding: 24px; background: #F9F7F4; border-radius: 12px; border: 1px solid #EEEDE9; box-sizing: border-box; max-width: 1100px;">';
        html += '<div style="display: flex; justify-content: space-between; align-items: center; margin-bottom: 20px;">';
        html += '<div style="font-size: 14pt; font-weight: 700; color: #0b2026;">Fastest \u2192 Slowest</div>';
        html += '<div style="font-size: 14pt; color: #618794;">Click a card to flip it</div>';
        html += '</div>';

        html += '<div style="display: grid; grid-template-columns: 1fr 1fr; gap: 20px;">';

        cards.forEach(function(card, index) {
            var isFlipped = flippedCards[index] === true;
            var backContent = document.getElementById(card.backId).innerHTML;

            html += '<div class="udfFlipCard" data-index="' + index + '" style="cursor: pointer; perspective: 1000px; height: 420px;">';
            html += '<div class="udf-card-inner" style="position: relative; width: 100%; height: 100%; transition: transform 0.6s; transform-style: preserve-3d;' + (isFlipped ? ' transform: rotateY(180deg);' : '') + '">';

            // Front
            html += '<div style="position: absolute; width: 100%; height: 100%; backface-visibility: hidden; -webkit-backface-visibility: hidden; background: #FFFFFF; border: 2px solid #EEEDE9; border-top: 6px solid ' + card.color + '; border-radius: 10px; padding: 24px; box-sizing: border-box; display: flex; flex-direction: column;">';
            html += '<div style="display:flex;align-items:center;gap:12px;margin-bottom:14px;"><span style="width:36px;height:36px;border-radius:50%;background:' + card.color + ';color:#fff;font-size:16pt;font-weight:800;display:flex;align-items:center;justify-content:center;flex-shrink:0;">' + card.num + '</span><span style="background:' + card.color + '; color: white; padding: 4px 12px; border-radius: 20px; font-size: 12pt; font-weight: 600;">' + card.badge + '</span></div>';
            html += '<div style="font-size: 16pt; font-weight: 700; color: #0b2026; margin-bottom: 12px; line-height: 1.3;">' + card.title + '</div>';
            html += '<div style="flex: 1; font-size: 14pt; color: #0b2026; line-height: 1.5;">' + card.subtitle + '</div>';
            html += '<div style="text-align: center; color: #618794; font-size: 13pt; margin-top: 12px;">Click to reveal code</div>';
            html += '</div>';

            // Back
            html += '<div style="position: absolute; width: 100%; height: 100%; backface-visibility: hidden; -webkit-backface-visibility: hidden; transform: rotateY(180deg); background: #FFFFFF; border: 2px solid ' + card.color + '; border-radius: 10px; padding: 20px; box-sizing: border-box; display: flex; flex-direction: column; overflow-y: auto;">';
            html += '<div style="margin-bottom: 10px;"><span style="background:' + card.color + '; color: white; padding: 4px 12px; border-radius: 20px; font-size: 12pt; font-weight: 600;">' + card.num + '. ' + card.title + '</span></div>';
            html += '<div style="flex: 1; font-size: 13pt; color: #0b2026; line-height: 1.5; overflow-y: auto;">' + backContent + '</div>';
            html += '<div style="text-align: center; color: #618794; font-size: 13pt; margin-top: 8px; flex-shrink: 0;">Click to flip back</div>';
            html += '</div>';

            html += '</div>';
            html += '</div>';
        });

        html += '</div>';

        html += '<div style="margin-top: 20px; padding-top: 16px; border-top: 2px solid #EEEDE9; display: flex; justify-content: space-between; align-items: center; flex-wrap: wrap; gap: 10px;">';
        html += '<div style="color: #0b2026; font-size: 14pt; line-height: 1.5;"><strong>In practice, use options 1 or 2.</strong> Try a built-in first. If you genuinely need Python, use an Arrow-optimized UDF. Only options 2 through 4 leave Photon.</div>';
        html += '<button id="udfResetBtn" style="padding: 10px 20px; background: #1B5162; color: white; border: none; border-radius: 6px; cursor: pointer; font-size: 14pt; font-weight: 600; flex-shrink: 0;">Reset All</button>';
        html += '</div>';

        html += '</div>';
        root.innerHTML = html;
        attachEvents();
    }

    function attachEvents() {
        var flipCards = document.querySelectorAll('.udfFlipCard');
        var resetBtn = document.getElementById('udfResetBtn');

        flipCards.forEach(function(card) {
            card.addEventListener('click', function() {
                var index = parseInt(this.dataset.index);
                flippedCards[index] = !flippedCards[index];
                var inner = this.querySelector('.udf-card-inner');
                inner.style.transform = flippedCards[index] ? 'rotateY(180deg)' : 'rotateY(0deg)';
            });
            card.addEventListener('mouseenter', function() {
                this.querySelector('.udf-card-inner').style.boxShadow = '0 8px 24px rgba(27,49,57,0.12)';
            });
            card.addEventListener('mouseleave', function() {
                this.querySelector('.udf-card-inner').style.boxShadow = 'none';
            });
        });

        resetBtn.addEventListener('click', function(e) {
            e.stopPropagation();
            flippedCards = {};
            document.querySelectorAll('.udf-card-inner').forEach(function(inner) {
                inner.style.transform = 'rotateY(0deg)';
            });
        });
        resetBtn.addEventListener('mouseenter', function() { this.style.background = '#0b2026'; });
        resetBtn.addEventListener('mouseleave', function() { this.style.background = '#1B5162'; });
    }

    if (typeof NodeList.prototype.forEach !== 'function') {
        NodeList.prototype.forEach = Array.prototype.forEach;
    }

    render();
})();
</script>

##### EXPAND FOR ADDITIONAL DETAILS

<details>

### Runtime nuance: serverless vs classic
- On serverless env v5, `spark.sql.execution.pythonUDF.arrow.enabled` is `true` and locked. A plain `udf(fn, returnType)` already uses Arrow batching (rung 2). You cannot fall to rung 4.
- On classic DBR 17.3, the same config defaults to `false`. A plain `udf(...)` is pickled row-at-a-time (rung 4) unless you set `useArrow=True`.
- Either way, a Python UDF sends the operator to the JVM instead of Photon. Arrow reduces serialization cost but does not restore Photon coverage. Only a built-in expression (option 1) stays fully in Photon.
- For this course we run the comparison between Python UDF (Arrow-batched, off Photon) vs built-in (fully in Photon) on serverless.

</details>




<div class="cw">
<style>
.cw { font-family: sans-serif; max-width: 1100px; margin: 0 auto; color: #0b2026; }
.cw * { box-sizing: border-box; }
.ds-callout { border-left: 6px solid var(--accent, #1B5162); border-radius: 8px; padding: 18px 24px; background: var(--tint, #F8F9FC); box-shadow: 0 2px 8px rgba(27,49,57,0.06); font-size: 14pt; line-height: 1.55; color: #0b2026; }
.ds-eyebrow { font-size: 14pt; font-weight: 700; letter-spacing: 1.5px; text-transform: uppercase; color: var(--accent, #1B5162); margin-bottom: 6px; }
.ds-callout-title { font-size: 16pt; font-weight: 700; color: #0b2026; margin-bottom: 10px; line-height: 1.3; }
.ds-callout ul { margin: 8px 0 0; padding-left: 20px; font-size: 14pt; line-height: 1.55; color: #0b2026; }
.ds-callout li { margin-bottom: 10px; } .ds-callout li:last-child { margin-bottom: 0; }
.ds-callout code { background: #EEEDE9; padding: 2px 6px; border-radius: 3px; font-size: 14pt; color: #0b2026; font-family: Menlo, Consolas, monospace; }
.caution { --accent:#FFAB00; --tint:rgba(255,171,0,0.12); }
</style>

<div class="ds-callout caution">
  <div class="ds-eyebrow">Platform behavior</div>
  <div class="ds-callout-title">Arrow is default-on (and locked) on serverless env v5</div>
  <ul>
    <li>On serverless, <code>spark.sql.execution.pythonUDF.arrow.enabled</code> is <code>true</code> and cannot be changed. A plain <code>udf(fn, returnType)</code> already uses Arrow batching.</li>
    <li>On classic DBR 17.3, the same config defaults to <code>false</code>. A plain <code>udf(...)</code> is pickled row-at-a-time unless you set <code>useArrow=True</code>.</li>
    <li>Either way, a Python UDF sends the operator to the JVM instead of Photon. Arrow reduces serialization cost but does not restore Photon coverage. Only a built-in expression (option 1) stays fully in Photon.</li>
  </ul>
</div>
</div>

##### EXPAND FOR ADDITIONAL DETAILS

<details>

### Serverless vs classic
- On serverless env v5, Arrow is locked on. The config throws `CONFIG_NOT_AVAILABLE` if you try to change it.
- On classic DBR 17.3 for example, the config defaults to `false`. Set `useArrow=True` per UDF, or flip the session config.
- For this course we'll run the comparison between Python UDF (Arrow-batched, off Photon) vs built-in (fully in Photon).

</details>

<div style="font-family:sans-serif; max-width:1100px; margin:0 auto; color:#0b2026;">
<div style="display:flex; align-items:center; gap:20px; width:100%; border-radius:10px; padding:24px 28px; background:#1B5162; border-left:8px solid #FF5F46; box-shadow:0 2px 10px rgba(27,49,57,0.08);">
  <div style="width:60px; height:60px; border-radius:50%; flex-shrink:0; background:#FF5F46; color:#FFFFFF; font-size:26pt; font-weight:700; line-height:1; display:flex; align-items:center; justify-content:center;">C</div>
  <div>
    <div style="font-size:22pt; font-weight:700; color:#FFFFFF; line-height:1.2; margin-bottom:4px;">1 transform, 2 implementations</div>
    <div style="font-size:15pt; color:#F9F7F4; line-height:1.4;">Python UDF vs built-in expression. Same answer, different Photon coverage.
    </div>
  </div>
</div>
</div>

### C. The Same Transform, Two Ways

The setup joined **order_items** to **products** (`base_line_items()`), about 60M rows. For each line item we derive a **tier label** like `ELE-A` from the product category and list price, then `GROUP BY` that label so the transform has to run on every row.

The shared Python logic is `price_band_py(category, list_price)` and the matching Spark expression is `builtin_tier_col()`. Both variants below compute the identical label, so the only thing that changes is how the work runs.

On serverless env v5, Arrow is locked on, so the Python UDF uses Arrow batching automatically. 

The comparison is: Python UDF (Arrow-batched, off Photon) vs built-in expression (fully in Photon).

In [ ]:
# Python UDF. On serverless, Arrow is default-on so serialization is batched, but the UDF still sends the operator to the JVM instead of Photon.
from pyspark.sql.functions import udf
from pyspark.sql.types import StringType

tier_udf = udf(price_band_py, StringType())

udf_df = (
    base_line_items()
    .withColumn("tier", tier_udf("category", "list_price"))
    .groupBy("tier")
    .count()
)

print("Variant 1: Python UDF (open its Query Profile after this cell runs) ...")
time_count(udf_df, "1. Python UDF")

In [ ]:
# No Python at all, stays in Photon. 
# builtin_tier_col() is the identical logic as a Spark expression.
builtin_df = (
    base_line_items()
    .withColumn("tier", builtin_tier_col())
    .groupBy("tier")
    .count()
)

print("Variant 2: built-in expression (open its Query Profile) ...")
time_count(builtin_df, "2. built-in expression")

In [ ]:
# Same transform should show the same result.
print("Tier-group counts (must match across both variants):")
print(f"  udf     : {udf_df.count()}")
print(f"  builtin : {builtin_df.count()}")

### C1. Read the Two Profiles
<br/>
<div class="cw">
<style>
.cw { font-family: sans-serif; max-width: 1100px; margin: 0 auto; color: #0b2026; display: flex; flex-direction: column; gap: 20px; }
.cw * { box-sizing: border-box; }
.ds-row { display: flex; gap: 24px; align-items: stretch; flex-wrap: wrap; }
.ds-row > * { flex: 1; min-width: 240px; }
.ds-card { background: #F9F7F4; border-radius: 8px; box-shadow: 0 2px 8px rgba(27,49,57,0.06); padding: 20px; border-top: 8px solid var(--accent, #4299E0); }
.blue{--accent:#4299E0;} .green{--accent:#00A972;} .coral{--accent:#FF5F46;} .amber{--accent:#FFAB00;} .red{--accent:#98102A;}
.ds-card-label { font-size: 14pt; font-weight: 700; letter-spacing: 1px; text-transform: uppercase; color: var(--accent); margin-bottom: 4px; }
.ds-card-title { font-size: 17pt; font-weight: 700; line-height: 1.25; margin: 0 0 8px; }
.ds-card ul { margin: 8px 0 0; padding-left: 20px; font-size: 14pt; line-height: 1.5; color: #0b2026; }
.ds-card li { margin-bottom: 8px; }
.ds-card li:last-child { margin-bottom: 0; }
</style>

<div class="ds-row">
  <div class="ds-card red">
    <div class="ds-card-label">Variant 1</div>
    <div class="ds-card-title">Python UDF</div>
    <ul>
      <li><code>total_task_duration_ms</code> ~460K. The <strong>Arrow Eval Python</strong> operator dominates (7+ min). Scan still runs in Photon, but its contribution is tiny next to the UDF.</li>
      <li>Arrow batches the serialization (default on serverless), but the Python worker still runs off Photon.</li>
    </ul>
  </div>
  <div class="ds-card green">
    <div class="ds-card-label">Variant 2</div>
    <div class="ds-card-title">Built-in</div>
    <ul>
      <li><code>total_task_duration_ms</code> ~50K (about 10x less). All compute operators run in Photon. No Arrow Eval Python in the plan.</li>
      <li>No Python worker, no serialization.</li>
      <li>Wall time drops from ~60s to ~4s.</li>
    </ul>
  </div>
</div>

<div style="border-left:4px solid #4299E0;background:#F8F9FC;padding:14px 18px;border-radius:4px;">
  <strong style="display:block;color:#1B5162;margin-bottom:6px;font-size:15pt;">Your numbers will vary</strong>
  <div style="color:#0b2026;font-size:14pt;line-height:1.55;">Wall time shifts between runs with cache warmth and parallelism. The stable signal is <code>total_task_duration_ms</code> from <code>system_vw.query.history</code> (aggregated compute across all tasks). The UDF shows ~460K ms; the built-in shows ~50K ms. That 10x gap is the cost of leaving Photon for a Python worker.</div>
</div>
</div>

##### EXPAND FOR ADDITIONAL DETAILS

<details>

### What to open, in order
- Run the two variant cells, then open each one's Query Profile (click the execution time at the bottom of the cell output, or go to Query History and click "See query profile").
- In the profile, look at the **costliest operators**:
  - **Python UDF**: The costliest operator is **Arrow Eval Python**, consuming nearly all execution time. `total_task_duration_ms` in query history is ~460,000 ms.
  - **Built-in**: The costliest operators are Scan and Inner Join, all Photon-native. No Arrow Eval Python operator exists. `total_task_duration_ms` is ~50,000 ms (about 10x cheaper).

### Why the built-in wins even though it does the same math
- The math is identical. The difference is entirely in where it runs. The built-in stays in Photon's vectorized C++ engine. The Python UDF crosses into a Python worker (via Arrow batches on serverless), which is cheaper than pickled serialization but still sends the operator to the JVM instead of Photon.

</details>

### C2. The Cost

Wall time varies between runs depending on cluster warm-up and scheduling. The more stable cost signal is `total_task_duration_ms` (aggregated compute across all tasks) from Part 2. 

The helper below reads it from `system_vw.query.history` for your recent runs. 

Read the rows in run order: UDF, then built-in.

There's a 5 to 10 minute ingestion lag. If a variant is missing, wait and re-run this cell.

In [ ]:
variant_cost_history()

## D. When a Query Leaves Photon
<br/>
<div class="phx">
  <style>
    .phx { font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", Roboto, Helvetica, Arial, sans-serif; color: #1f2933; background: #ffffff; padding: 18px 16px 22px 16px; border: 1px solid #e2e8ee; border-radius: 10px; max-width: 1040px; margin: 0 auto; }
    .phx h3 { margin: 0 0 4px 0; font-size: 19px; font-weight: 650; color: #14343f; letter-spacing: .1px; }
    .phx p.sub { margin: 0 0 14px 0; font-size: 15px; color: #5f6b76; }
    .phx svg { display: block; width: 100%; height: auto; }
    .phx text { font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", Roboto, Helvetica, Arial, sans-serif; }
    .phx .hdr { font-size: 14px; font-weight: 700; fill: #14343f; letter-spacing: .3px; }
    .phx .op { font-size: 13px; font-weight: 700; fill: #ffffff; }
    .phx .sm { font-size: 12px; fill: #62707d; }
    .phx .tag { font-size: 11px; font-weight: 700; letter-spacing: .4px; }
    .phx .note { margin-top: 12px; font-size: 14px; color: #5f6b76; border-left: 3px solid #0f9aab; padding: 4px 0 4px 10px; line-height: 1.55; }
    .phx code { font-family: ui-monospace, SFMono-Regular, Menlo, Consolas, monospace; font-size: 14px; background: #f2f5f8; padding: 1px 4px; border-radius: 3px; color: #23424c; }
  </style>
  <h3>What Leaving Photon Looks Like in the Profile</h3>
  <p class="sub">Photon runs built-in operators in its vectorized C++ engine. It cannot run a Python UDF, so it hands that operator to the JVM. The profile shows exactly where.</p>
  <svg viewBox="0 0 1040 300" role="img" aria-label="A query plan where scan, filter, and aggregate run in Photon (purple), but the Python UDF operator leaves Photon for the JVM (grey, dashed).">
    <defs>
      <marker id="phx-arr" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="7" markerHeight="7" orient="auto-start-reverse">
        <path d="M 0 0 L 10 5 L 0 10 z" fill="#5b6b78"></path>
      </marker>
    </defs>
    <text class="hdr" x="16" y="28">ONE QUERY PLAN, read left to right</text>
    <!-- operator chain: purple = photon, grey = leaves photon -->
    <!-- scan (photon) -->
    <rect x="16" y="48" width="150" height="64" rx="8" fill="#7b5ea7"></rect>
    <text class="op" x="91" y="76" text-anchor="middle">Scan</text>
    <text class="sm" style="fill:#ffffff" x="91" y="96" text-anchor="middle">Photon</text>
    <line x1="166" y1="80" x2="196" y2="80" stroke="#5b6b78" stroke-width="1.8" marker-end="url(#phx-arr)"></line>
    <!-- filter (photon) -->
    <rect x="198" y="48" width="150" height="64" rx="8" fill="#7b5ea7"></rect>
    <text class="op" x="273" y="76" text-anchor="middle">Filter</text>
    <text class="sm" style="fill:#ffffff" x="273" y="96" text-anchor="middle">Photon</text>
    <line x1="348" y1="80" x2="378" y2="80" stroke="#5b6b78" stroke-width="1.8" marker-end="url(#phx-arr)"></line>
    <!-- UDF (grey fallback) -->
    <rect x="380" y="42" width="180" height="76" rx="8" fill="#8a97a3" stroke="#5b6b78" stroke-width="2" stroke-dasharray="5 4"></rect>
    <text class="op" x="470" y="70" text-anchor="middle">Python UDF</text>
    <text class="sm" style="fill:#ffffff" x="470" y="90" text-anchor="middle">NOT Photon (JVM)</text>
    <text class="tag" x="470" y="108" text-anchor="middle" fill="#ffffff">LEAVES PHOTON</text>
    <line x1="560" y1="80" x2="590" y2="80" stroke="#5b6b78" stroke-width="1.8" marker-end="url(#phx-arr)"></line>
    <!-- aggregate (photon) -->
    <rect x="592" y="48" width="150" height="64" rx="8" fill="#7b5ea7"></rect>
    <text class="op" x="667" y="76" text-anchor="middle">Aggregate</text>
    <text class="sm" style="fill:#ffffff" x="667" y="96" text-anchor="middle">Photon</text>
    <!-- coverage insight callout -->
    <rect x="770" y="42" width="254" height="76" rx="8" fill="#fdf3ec" stroke="#e5bfa2"></rect>
    <text class="tag" x="786" y="64" fill="#a85f2c">PERFORMANCE INSIGHT</text>
    <text class="op" x="786" y="86" style="fill:#7a4a2e">COVERAGE_PHOTON</text>
    <text class="sm" style="fill:#7a4a2e" x="786" y="106">names the exit for you</text>
    <!-- legend -->
    <rect x="16" y="140" width="18" height="18" rx="3" fill="#7b5ea7"></rect>
    <text class="sm" x="42" y="154">Purple operators run in Photon</text>
    <rect x="300" y="140" width="18" height="18" rx="3" fill="#8a97a3"></rect>
    <text class="sm" x="326" y="154">Grey operators left Photon for the JVM</text>
    <text class="sm" x="620" y="154">Execution Details shows the percent of task time in Photon</text>
    <!-- three-part guidance strip -->
    <rect x="16" y="176" width="330" height="108" rx="8" fill="#fef2f4" stroke="#d4808f"></rect>
    <text class="tag" x="32" y="200" fill="#98102A">WHAT FORCES IT</text>
    <text class="sm" x="32" y="222">Python UDFs (pickled and Arrow),</text>
    <text class="sm" x="32" y="240">Scala and Java UDFs, and a few</text>
    <text class="sm" x="32" y="258">operators Photon does not cover yet.</text>
    <rect x="356" y="176" width="330" height="108" rx="8" fill="#eef6fd" stroke="#9fcdd6"></rect>
    <text class="tag" x="372" y="200" fill="#0f6b7a">READ IT IN THE PROFILE</text>
    <text class="sm" x="372" y="222">total_task_duration_ms in query history,</text>
    <text class="sm" x="372" y="240">Arrow Eval Python as costliest</text>
    <text class="sm" x="372" y="258">operator, COVERAGE_PHOTON insight.</text>
    <rect x="696" y="176" width="328" height="108" rx="8" fill="#f0faf5" stroke="#8bbf9c"></rect>
    <text class="tag" x="712" y="200" fill="#2d6e3f">WHAT TO DO</text>
    <text class="sm" x="712" y="222">Replace the UDF with a built-in</text>
    <text class="sm" x="712" y="240">to stay in Photon.</text>
    <text class="sm" x="712" y="258">Re-check the Photon ratio after.</text>
  </svg>
  <div class="note">The insights from Part 7 (DATA_SKEW, DATA_SPILL, EXPLODING_JOIN) were about distributed work under stress. <code>COVERAGE_PHOTON</code> is a different cost: the work is still distributed across tasks, but the UDF operator leaves Photon for the slower JVM engine and every row crosses into a Python worker. Section C produced these profiles for the Python UDF and built-in variants.</div>
</div>

##### EXPAND FOR ADDITIONAL DETAILS

<details>

### Why a UDF leaves Photon
- Photon is a vectorized C++ engine. It executes built-in operators over columnar batches. It cannot execute an arbitrary Python (or Scala or Java) function body, so when a UDF appears in the plan, Photon runs everything it can and hands the UDF operator to the JVM engine.
- This connects Part 8 back to Part 7's spill and shuffle story. Those were about distributed work under stress. Leaving Photon is a different cost: the work is still distributed across tasks, but the UDF portion runs in the slower engine, and every row crosses into a Python worker.

### Reading it in the Query Profile
- Open the profile and look at the costliest operators and `total_task_duration_ms` in query history. A query with Python UDFs shows Arrow Eval Python dominating (the UDF variant: ~460K ms task time vs ~50K ms for built-in).
- The costliest operator will be **Arrow Eval Python**, consuming nearly all execution time. This is the Photon exit made visible.
- The **COVERAGE_PHOTON** Performance Insight is the named signal. It sits in the same family as the DATA_SKEW, DATA_SPILL, and EXPLODING_JOIN insights from Part 7. When it fires, part of your query left Photon for the JVM.
- Section C already produced these profiles. The UDF variant shows Arrow Eval Python dominating at ~460K ms task time. The built-in shows ~50K ms with no Python operator. The UDF choice determines whether the query stays in Photon.

</details>

<div style="width: 100%; font-family: sans-serif;">
<div style="background: #F9F7F4; border-radius: 10px; padding: 24px 28px; box-shadow: 0 2px 8px rgba(27,49,57,0.06); border-top: 6px solid #FF5F46;">
  <div style="margin-bottom: 10px;"><svg viewBox="0 0 806 200" fill="none" xmlns="http://www.w3.org/2000/svg" style="height:64px; width:auto; display:block;"><path d="M81.7909 159.789C75.0244 159.789 69.5392 166.599 69.5392 175H126.368C126.368 166.599 120.883 159.789 114.116 159.789H81.7909Z" fill="#FF5F46"/><path d="M179.614 80.7871C167.525 100.208 149.376 126.933 141.272 133.79C133.167 140.647 121.149 153.009 93.2907 153.009C73.7741 153.009 56.9864 140.474 49.5352 122.498C49.5353 122.519 49.5356 122.54 49.5358 122.561C48.4799 119.938 45.9132 118.085 42.9121 118.085C38.969 118.085 35.7721 121.282 35.7721 125.225C35.7722 126.15 35.9484 126.997 36.2689 127.752C37.984 131.793 42.5469 131.965 42.5469 144.569C42.5469 143.103 42.5469 146.404 42.5469 144.569C30.6947 144.569 21.0865 134.96 21.0863 123.108C21.0863 111.256 30.6945 101.647 42.5469 101.647H45.4249V101.644H79.5153C82.062 101.644 84.5067 102.722 86.0978 104.71C88.8856 108.194 91.0222 111.438 92.5969 114.114C94.056 116.594 98.4255 116.618 99.8699 114.13C101.393 111.506 103.416 108.337 105.98 104.929C107.573 102.813 110.103 101.644 112.752 101.644H125.427C143.951 101.644 154.105 92.7623 159.289 86.3612C161.972 83.049 165.782 80.7871 170.045 80.7871H179.614Z" fill="#FABFBA"/><path d="M96.3843 111.768C96.3842 87.8629 77.3194 68.4739 53.768 68.3843C77.3196 68.2949 96.3845 48.9049 96.3845 25C96.3845 25 96.3845 25 96.3845 25C96.3845 48.9602 115.538 68.3838 139.165 68.3838C115.538 68.3838 96.3844 87.8075 96.3843 111.768Z" fill="#FF5F46"/><path d="M274.824 139.867C267.307 139.867 260.701 138.158 255.006 134.742C249.388 131.325 245.022 126.579 241.909 120.505C238.796 114.355 237.239 107.18 237.239 98.9793C237.239 90.855 238.834 83.6798 242.023 77.4537C245.212 71.1516 249.767 66.2543 255.69 62.7615C261.688 59.1929 268.787 57.4086 276.988 57.4086C286.251 57.4086 293.92 59.6485 299.994 64.1283C306.144 68.5321 310.13 74.6823 311.952 82.5788H296.577C295.438 78.6306 293.198 75.5555 289.857 73.3535C286.516 71.1516 282.189 70.0507 276.874 70.0507C271.483 70.0507 266.851 71.2276 262.979 73.5813C259.182 75.8592 256.259 79.2 254.209 83.6039C252.235 87.9318 251.248 93.0949 251.248 99.0932C251.248 105.092 252.235 110.255 254.209 114.583C256.259 118.911 259.144 122.213 262.865 124.491C266.585 126.693 270.951 127.794 275.962 127.794C283.403 127.794 289.022 125.706 292.819 121.53C296.615 117.354 298.855 111.925 299.538 105.243H278.924V94.8792H313.319V138.5H300.791L299.652 127.566C297.906 130.148 295.894 132.388 293.616 134.286C291.338 136.108 288.642 137.513 285.529 138.5C282.492 139.411 278.924 139.867 274.824 139.867ZM351.07 139.867C345.375 139.867 340.364 138.652 336.036 136.222C331.708 133.717 328.33 130.224 325.9 125.744C323.47 121.264 322.255 116.101 322.255 110.255C322.255 104.18 323.432 98.8654 325.786 94.3097C328.216 89.7541 331.594 86.1854 335.922 83.6039C340.326 81.0223 345.413 79.7315 351.184 79.7315C356.803 79.7315 361.7 80.9843 365.876 83.49C370.052 85.9956 373.279 89.3744 375.557 93.6264C377.835 97.8024 378.974 102.51 378.974 107.749C378.974 108.508 378.974 109.344 378.974 110.255C378.974 111.166 378.898 112.115 378.746 113.102H332.05V104.332H365.193C365.041 100.156 363.636 96.8913 360.979 94.5375C358.321 92.1078 355.018 90.893 351.07 90.893C348.261 90.893 345.679 91.5384 343.325 92.8291C340.972 94.1199 339.111 96.0561 337.745 98.6377C336.378 101.143 335.695 104.332 335.695 108.205V111.507C335.695 115.076 336.34 118.151 337.631 120.733C338.997 123.314 340.82 125.288 343.098 126.655C345.451 127.946 348.071 128.591 350.956 128.591C354.145 128.591 356.765 127.908 358.815 126.541C360.941 125.175 362.497 123.352 363.484 121.074H377.379C376.316 124.643 374.57 127.87 372.14 130.755C369.71 133.565 366.711 135.805 363.143 137.475C359.574 139.069 355.55 139.867 351.07 139.867ZM388.176 138.5V81.0982H400.248L401.273 90.6652C403.02 87.3243 405.525 84.6669 408.79 82.6927C412.055 80.7186 415.928 79.7315 420.407 79.7315C425.039 79.7315 428.987 80.7186 432.252 82.6927C435.517 84.5909 438.023 87.4003 439.769 91.1208C441.591 94.8412 442.503 99.4729 442.503 105.016V138.5H428.835V106.268C428.835 101.485 427.772 97.8024 425.646 95.2209C423.52 92.6393 420.369 91.3485 416.193 91.3485C413.46 91.3485 410.992 91.9939 408.79 93.2847C406.664 94.5755 404.956 96.4737 403.665 98.9793C402.45 101.409 401.843 104.37 401.843 107.863V138.5H388.176ZM454.064 138.5V81.0982H467.731V138.5H454.064ZM460.898 71.9868C458.392 71.9868 456.304 71.2275 454.634 69.709C453.039 68.1145 452.242 66.1783 452.242 63.9005C452.242 61.5467 453.039 59.6485 454.634 58.2059C456.304 56.6873 458.392 55.928 460.898 55.928C463.404 55.928 465.454 56.6873 467.048 58.2059C468.719 59.6485 469.554 61.5467 469.554 63.9005C469.554 66.1783 468.719 68.1145 467.048 69.709C465.454 71.2275 463.404 71.9868 460.898 71.9868ZM506.987 139.867C501.292 139.867 496.281 138.652 491.953 136.222C487.625 133.717 484.246 130.224 481.817 125.744C479.387 121.264 478.172 116.101 478.172 110.255C478.172 104.18 479.349 98.8654 481.703 94.3097C484.133 89.7541 487.511 86.1854 491.839 83.6039C496.243 81.0223 501.33 79.7315 507.101 79.7315C512.72 79.7315 517.617 80.9843 521.793 83.49C525.969 85.9956 529.196 89.3744 531.474 93.6264C533.752 97.8024 534.891 102.51 534.891 107.749C534.891 108.508 534.891 109.344 534.891 110.255C534.891 111.166 534.815 112.115 534.663 113.102H487.967V104.332H521.11C520.958 100.156 519.553 96.8913 516.896 94.5375C514.238 92.1078 510.935 90.893 506.987 90.893C504.178 90.893 501.596 91.5384 499.242 92.8291C496.889 94.1199 495.028 96.0561 493.662 98.6377C492.295 101.143 491.612 104.332 491.612 108.205V111.507C491.612 115.076 492.257 118.151 493.548 120.733C494.914 123.314 496.737 125.288 499.015 126.655C501.368 127.946 503.988 128.591 506.873 128.591C510.062 128.591 512.682 127.908 514.732 126.541C516.858 125.175 518.414 123.352 519.401 121.074H533.296C532.233 124.643 530.487 127.87 528.057 130.755C525.627 133.565 522.628 135.805 519.06 137.475C515.491 139.069 511.467 139.867 506.987 139.867Z" fill="#0B2026"/><rect x="563.157" y="58.4648" width="242.843" height="84.9055" rx="42.4527" fill="#0B2026"/><path d="M664.499 114.947C661.815 114.947 659.493 114.366 657.532 113.205C655.596 112.018 654.1 110.38 653.042 108.289C652.01 106.173 651.493 103.735 651.493 100.974C651.493 98.187 652.01 95.7485 653.042 93.6584C654.1 91.5424 655.596 89.8909 657.532 88.7039C659.493 87.517 661.815 86.9235 664.499 86.9235C667.724 86.9235 670.356 87.7234 672.395 89.3232C674.433 90.8973 675.724 93.1294 676.266 96.0195H671.156C670.795 94.497 670.06 93.2971 668.95 92.4198C667.84 91.5424 666.344 91.1037 664.46 91.1037C662.757 91.1037 661.286 91.5037 660.048 92.3036C658.835 93.0778 657.893 94.2132 657.222 95.7098C656.577 97.1807 656.254 98.9353 656.254 100.974C656.254 103.012 656.577 104.767 657.222 106.238C657.893 107.683 658.835 108.805 660.048 109.605C661.286 110.38 662.757 110.767 664.46 110.767C666.344 110.767 667.84 110.367 668.95 109.567C670.06 108.741 670.795 107.606 671.156 106.161H676.266C675.749 108.896 674.459 111.05 672.395 112.625C670.356 114.173 667.724 114.947 664.499 114.947ZM697.803 114.947C695.196 114.947 692.887 114.366 690.874 113.205C688.887 112.018 687.326 110.38 686.191 108.289C685.055 106.173 684.488 103.722 684.488 100.935C684.488 98.1741 685.055 95.7485 686.191 93.6584C687.326 91.5424 688.887 89.8909 690.874 88.7039C692.887 87.517 695.196 86.9235 697.803 86.9235C700.46 86.9235 702.796 87.517 704.808 88.7039C706.821 89.8909 708.382 91.5424 709.492 93.6584C710.602 95.7485 711.156 98.1741 711.156 100.935C711.156 103.722 710.602 106.173 709.492 108.289C708.382 110.38 706.821 112.018 704.808 113.205C702.796 114.366 700.46 114.947 697.803 114.947ZM697.841 110.767C699.57 110.767 701.08 110.38 702.37 109.605C703.66 108.805 704.654 107.67 705.35 106.199C706.047 104.728 706.395 102.974 706.395 100.935C706.395 98.8966 706.047 97.1419 705.35 95.6711C704.654 94.2003 703.66 93.0778 702.37 92.3036C701.08 91.5037 699.57 91.1037 697.841 91.1037C696.112 91.1037 694.603 91.5037 693.313 92.3036C692.022 93.0778 691.016 94.2003 690.294 95.6711C689.597 97.1419 689.249 98.8966 689.249 100.935C689.249 102.974 689.597 104.728 690.294 106.199C691.016 107.67 692.022 108.805 693.313 109.605C694.603 110.38 696.112 110.767 697.841 110.767ZM720.339 114.482V87.3879H729.087C732.235 87.3879 734.829 87.9427 736.867 89.0523C738.931 90.1619 740.454 91.7359 741.434 93.7745C742.441 95.7872 742.944 98.187 742.944 100.974C742.944 103.709 742.441 106.096 741.434 108.135C740.454 110.147 738.931 111.708 736.867 112.818C734.829 113.928 732.235 114.482 729.087 114.482H720.339ZM724.984 110.534H728.855C731.203 110.534 733.048 110.16 734.39 109.412C735.757 108.638 736.725 107.541 737.293 106.122C737.886 104.677 738.183 102.961 738.183 100.974C738.183 98.9611 737.886 97.2452 737.293 95.8259C736.725 94.3809 735.757 93.2713 734.39 92.4972C733.048 91.6972 731.203 91.2973 728.855 91.2973H724.984V110.534ZM752.124 114.482V87.3879H769.619V91.1425H756.769V98.9224H768.458V102.6H756.769V110.728H769.619V114.482H752.124Z" fill="white"/><path d="M636.472 101.239L620.99 116.72L613.554 109.283L621.598 101.239L613.554 93.1943L620.99 85.7578L636.472 101.239Z" fill="#FF3621"/><path d="M584.879 101.241L600.36 85.7598L607.797 93.1968L599.753 101.241L607.797 109.286L600.36 116.723L584.879 101.241Z" fill="#FF3621"/></svg></div>
  <div style="font-size: 15pt; color: #0b2026; line-height: 1.7; margin-bottom: 16px;">When you find driver-bound code in codebases, select it and use <strong>/optimize</strong> in Genie Code.</div>
</div>
</div>

##### EXPAND FOR ADDITIONAL DETAILS

<details>

### What to check in any Genie Code rewrite
- Does it remove `collect()` and `toPandas()`?
- Does it keep the math as Spark expressions (not pandas on the driver)?
- Does the profile show work spread across tasks, not a single-thread driver stretch?
- Is `total_task_duration_ms` lower than the original?

</details>

## Conclusion

Parts 1 through 7 covered distributed slow-downs: shuffle, skew, spill, and explosion. This part covered the fifth pattern: work that never reaches the executors, or that leaves Photon when it gets there.

1. **Driver versus executor**: the driver plans and coordinates. Code that pulls data to the driver with `collect()` or `toPandas()` runs single-threaded while the cluster sits idle. The profile looks quiet because the heavy work happened in Python on the driver, off the engine entirely.
2. **Driver traps**: `collect()`, `toPandas()`, single-threaded pandas, `display()` and repeated `count()` in production, and plan bloat. The fix is to rewrite the code so the work stays distributed, not to up-size the driver.
3. **UDF options**: built-in expressions stay fully in Photon. A Python UDF sends its operator to the JVM instead. On serverless, Arrow is locked on, so the comparison is Python UDF (off Photon, ~460K ms task time) vs built-in (in Photon, ~50K ms). That 10x gap is the cost of leaving the engine.
4. **Reading it in the profile**: the **Arrow Eval Python** operator dominates task time for the UDF variant. `total_task_duration_ms` in `system_vw.query.history` is the stable, cache-neutral metric that confirms the difference.


&copy; <span id="dbx-year"></span> Databricks, Inc. All rights reserved.
Apache, Apache Spark, Spark, the Spark Logo, Apache Iceberg, Iceberg, and the Apache Iceberg logo are trademarks of the <a href="https://www.apache.org/" target="_blank">Apache Software Foundation</a>.<br/><br/><a href="https://databricks.com/privacy-policy" target="_blank">Privacy Policy</a> | <a href="https://databricks.com/terms-of-use" target="_blank">Terms of Use</a> | <a href="https://help.databricks.com/" target="_blank">Support</a>
<script>
  document.getElementById("dbx-year").textContent = new Date().getFullYear();
</script>